# Data preparation

We will reorganize the files so is easy to work with for the training of our model

## Import libraries

In [1]:
import random
import shutil
import zipfile

import numpy as np
import tifffile
import torch
from torch.utils.data import DataLoader, Dataset

from zenodo_get import download
from pathlib import Path

## Pull dataset from Zenodo

In [2]:
# Folder that holds A172.zip; the extracted A172/ folder will be created here
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

if not any(data_dir.rglob("A*.zip")):
        download(
                record_or_doi="10277106",
                output_dir=str(data_dir),
                file_glob="A*.zip",
        )
else:
        print("An A*.zip file already exists; skipping download.")

An A*.zip file already exists; skipping download.


## Extract and reorganize the dataset

In [3]:
# Extract only if the dataset is missing or contains no TIFF files.
cell_type = "A172"
root = data_dir / cell_type
zip_path = data_dir / f"{cell_type}.zip"

if not root.exists() or next(root.rglob("*.tif"), None) is None:
    if not zip_path.is_file():
        raise FileNotFoundError(f"Dataset archive not found: {zip_path}")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(data_dir)
    if next(root.rglob("*.tif"), None) is None:
        raise FileNotFoundError(f"No .tif files were extracted under {root}")
    print(f"Extracted {zip_path} to {root}")
else:
    print(f"{root} already contains TIFF files; skipping extraction.")

../data/A172 already contains TIFF files; skipping extraction.


### Data structure
This is the original source layout. The following shell commands display the directory tree and count the images in each original split:


In [4]:
!tree ../data/ -L 3

../data/
├── A172
│   ├── test
│   │   ├── input
│   │   └── output
│   └── train
│       ├── input
│       └── output
└── A172.zip

7 directories, 1 file


In [5]:
! ls ../data/A172/test/input | wc -l

152


In [6]:
! ls ../data/A172/train/input | wc -l

456


The archive contains `train/` and `test/` source splits. Each has an `input/` folder of TIFF images and an `output/` folder of same-named ground-truth masks. We combine both source splits into one pool, make a seeded 80/10/10 sample split, then copy each image-mask pair into its own split folder. The original extracted files are left unchanged.

### Source dataset layout

The source archive contains the `train/` and `test/` directories shown above. We combine both into one pool before sampling, so the original test split is not kept as a separate test set.

### Collect matching images and annotations

In [7]:
# Match each image to its same-named mask across both source splits.
source_pairs = []
seen_image_names = set()

for split_name in ("train", "test"):
    input_dir = root / split_name / "input"
    output_dir = root / split_name / "output"
    if not input_dir.is_dir() or not output_dir.is_dir():
        raise FileNotFoundError(f"Expected input/ and output/ folders under {root / split_name}")

    image_files = sorted(input_dir.glob("*.tif"))
    if not image_files:
        raise FileNotFoundError(f"No .tif images found in {input_dir}")

    for image_path in image_files:
        mask_path = output_dir / image_path.name
        if not mask_path.is_file():
            raise FileNotFoundError(f"No matching annotation for {image_path}")
        if image_path.name in seen_image_names:
            raise ValueError(f"Duplicate image filename across source splits: {image_path.name}")
        seen_image_names.add(image_path.name)
        source_pairs.append((image_path, mask_path))

print(f"Collected {len(source_pairs)} matched image-mask pairs from the source splits")

Copied 608 image-mask pairs into ../data/A172_flat


## Make a reproducible 80/10/10 split

Set `N_IMAGES` to the number of image-mask pairs to sample. It must be a positive multiple of 10 so the train, test, and validation sets remain exactly 80%, 10%, and 10%. `SPLIT_SEED` controls which pairs are selected and keeps the split reproducible. Each pair is copied into one of three folders under a run-specific prepared-data root, with the image and `_masks` annotation together.

In [10]:
N_IMAGES = 100
SPLIT_SEED = 42

if N_IMAGES <= 0 or N_IMAGES % 10 != 0:
    raise ValueError("N_IMAGES must be a positive multiple of 10 for an exact 80/10/10 split")
if N_IMAGES > len(source_pairs):
    raise ValueError(f"Requested {N_IMAGES} pairs, but only {len(source_pairs)} are available")

selected_pairs = random.Random(SPLIT_SEED).sample(source_pairs, N_IMAGES)
n_train = N_IMAGES * 8 // 10
n_test = N_IMAGES // 10

train_pairs = selected_pairs[:n_train]
test_pairs = selected_pairs[n_train:n_train + n_test]
validation_pairs = selected_pairs[n_train + n_test:]

assert len(train_pairs) == n_train
assert len(test_pairs) == N_IMAGES // 10
assert len(validation_pairs) == N_IMAGES // 10

# Use a separate output root so the original A172/train and A172/test remain untouched.
prepared_dir = data_dir / f"{cell_type}_splits_n{N_IMAGES}_seed{SPLIT_SEED}"
split_groups = {
    "train": train_pairs,
    "test": test_pairs,
    "validation": validation_pairs,
}

for split_name, pairs in split_groups.items():
    split_dir = prepared_dir / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    copied_pairs = []
    for image_path, mask_path in pairs:
        split_image = split_dir / image_path.name
        split_mask = split_dir / f"{image_path.stem}_masks{image_path.suffix}"
        shutil.copy2(image_path, split_image)
        shutil.copy2(mask_path, split_mask)
        copied_pairs.append((split_image, split_mask))
    split_groups[split_name] = copied_pairs

train_pairs = split_groups["train"]
test_pairs = split_groups["test"]
validation_pairs = split_groups["validation"]

assert len(train_pairs) == n_train
assert len(test_pairs) == N_IMAGES // 10
assert len(validation_pairs) == N_IMAGES // 10
print(f"Copied {len(train_pairs)} train, {len(test_pairs)} test, and {len(validation_pairs)} validation pairs to {prepared_dir}")

Copied 80 train, 10 test, and 10 validation pairs to ../data/A172_splits_n100_seed42


## Create PyTorch datasets and data loaders

The custom `Dataset` reads one TIFF image and its matching instance mask at a time. Images are returned as normalized float tensors with shape `[1, height, width]`; masks stay as integer label tensors with shape `[height, width]`. The `DataLoader`s batch the data for a PyTorch segmentation model.

In [9]:
class A172Dataset(Dataset):
    def __init__(self, pairs):
        self.pairs = list(pairs)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]
        image = np.asarray(tifffile.imread(image_path))
        mask = np.asarray(tifffile.imread(mask_path))

        if image.ndim != 2 or mask.ndim != 2:
            raise ValueError(f"Expected grayscale 2D TIFFs: {image_path.name}, {mask_path.name}")
        if image.shape != mask.shape:
            raise ValueError(f"Image and mask dimensions differ for {image_path.name}")
        if not np.issubdtype(mask.dtype, np.integer):
            raise TypeError(f"Mask must contain integer labels: {mask_path}")

        if np.issubdtype(image.dtype, np.integer):
            image_scale = np.iinfo(image.dtype).max
        else:
            image_scale = 1.0
        image = image.astype(np.float32) / image_scale

        image_tensor = torch.from_numpy(np.ascontiguousarray(image)).unsqueeze(0)
        mask_tensor = torch.from_numpy(np.ascontiguousarray(mask, dtype=np.int64))
        return image_tensor, mask_tensor


train_dataset = A172Dataset(train_pairs)
test_dataset = A172Dataset(test_pairs)
validation_dataset = A172Dataset(validation_pairs)

batch_size = 4
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=torch.Generator().manual_seed(SPLIT_SEED),
)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

images, masks = next(iter(train_loader))
print(f"Image batch: {tuple(images.shape)} ({images.dtype})")
print(f"Mask batch: {tuple(masks.shape)} ({masks.dtype})")

Image batch: (4, 1, 520, 704) (torch.float32)
Mask batch: (4, 520, 704) (torch.int64)
